In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# %matplotlib qt
%matplotlib widget

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from dashsrc.plot_components.plot_wrappers.data_selection import group_filter_data

from analytics_processing.modality_loading import session_modality_from_nas
from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

# import GridSpecFromSubplotSpec
from matplotlib.gridspec import GridSpecFromSubplotSpec
import scipy.stats as stats
import statsmodels.api as sm
from matplotlib.colors import LinearSegmentedColormap, Normalize



In [ ]:
output_dir = "./outputs/experimental/"
data = {}
nas_dir = C.device_paths()[0]
Logger().init_logger(None, None, logging_level="WARNING")

In [ ]:
animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [ ]:
fr_raw = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
fr = fr_raw.drop("from_ephys_timestamp", axis=1)
fr.index = fr.index.droplevel((0,1,3, ))
fr.set_index('to_ephys_timestamp', append=True, inplace=True)
fr['global_t'] = np.arange(40_000, 40_000*len(fr)+1, 40_000)
fr.set_index('global_t', append=True, inplace=True)
fr.index = fr.index.rename(['session_id', 'session_t', 'global_t'])
session_t_start = fr.index[fr.index.get_level_values('session_t') == 40_000]

fr = fr.fillna(0)
fr_z = fr.apply(lambda unit_fr: ((unit_fr - unit_fr.mean()) / unit_fr.std()))

fr

In [ ]:
step_us = 60 * 1e6  # 1 minute in microseconds
window_size_us = 3 * step_us  # 3 minutes
end_time = int(fr.index.get_level_values('global_t').max()) - int(window_size_us)

all_sess_eigenvecs = []
for start_time in range(0, end_time, int(step_us)):
    end_time = start_time + window_size_us
    # print("From ", start_time, " to ", end_time)
    # continue
    window_data = fr_z.xs(slice(start_time, end_time), level='global_t')
    s_id = window_data.index.get_level_values('session_id').unique()[0]
    
    # print(window_data.index)
    window_Z = window_data.values.T  # transpose to have neurons in rows, time bins in columns
    const_neuron_indices = np.where((window_Z.var(axis=1)) > 1e-3)

    # compute correlation matrix
    corr_matrix = np.corrcoef(window_Z[const_neuron_indices])
    eigenvals, eigenvecs = np.linalg.eigh(corr_matrix)
    
    # Sort eigenvalues in descending order
    idx = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]
    
    # plt.figure()
    # plt.imshow(eigenvecs, aspect='auto', cmap='bwr', vmin=-1, vmax=1)
    # plt.show()
    
    full_eigenvecs = np.zeros((fr.shape[1], fr.shape[1]))
    # insert eigenvecs back to full size
    idx = const_neuron_indices[0]
    # this is a square operation, also introduces unwanted columns
    full_eigenvecs[np.ix_(idx, idx)] = eigenvecs
    # get rid of full 0 eigenvec columns, we only want the rows in original dim
    full_eigenvecs = full_eigenvecs[:, (full_eigenvecs != 0).any(axis=0)]
    # limit to 80 perc varianace explained
    eigenval_sum = np.sum(eigenvals)
    expl_var = eigenvals / eigenval_sum
    expl_var_cum = np.cumsum(expl_var)
    n_components_80 = np.where(expl_var_cum >= 0.8)[0][0] + 1
    # print("Number of components to explain 80% variance: ", n_components_80)
    full_eigenvecs = full_eigenvecs[:, :n_components_80].astype(np.float32)

    print(full_eigenvecs.T.shape)
    all_sess_eigenvecs.append(pd.DataFrame(full_eigenvecs.T, index=pd.MultiIndex.from_product(
        [[s_id], [start_time], np.arange(full_eigenvecs.shape[1])], names=['session_id', 'start_time', 'component_index'])))

    # plt.figure()
    # plt.imshow(full_eigenvecs, aspect='auto', cmap='bwr', vmin=-1, vmax=1)
    # plt.show()

    # print(full_eigenvecs.shape)
    # print(start_time / end_time)
    # print(full_eigenvecs.dtype)
    
    # if start_time > 100000000:
    #     break


# corr_matrix = np.corrcoef(Z)
# eigenvals, eigenvecs = np.linalg.eigh(corr_matrix)
    
# # Sort eigenvalues in descending order
# idx = np.argsort(eigenvals)[::-1]
# eigenvals = eigenvals[idx]
# eigenvecs = eigenvecs[:, idx]
# # eigenvecs
# eigenval_sum = np.sum(eigenvals)
# expl_var = eigenvals / eigenval_sum
# expl_var_cum = np.cumsum(expl_var)
# expl_var_cum

all_sess_eigenvecs = pd.concat(all_sess_eigenvecs)
all_sess_eigenvecs

In [ ]:
windows = all_sess_eigenvecs.index.unique('start_time')

angle_aggr = []
session_seps = []
last_session_id = None
for i, from_window in enumerate(windows):
    from_pca_subspace = all_sess_eigenvecs.xs(from_window, level='start_time').values.T
    session_id = all_sess_eigenvecs.xs(from_window, level='start_time').index.get_level_values('session_id')[0]
    if last_session_id is None:
        last_session_id = session_id
        session_seps.append(i)
    
    elif last_session_id != session_id:
        last_session_id = session_id
        session_seps.append(i)
    
        
    for to_window in windows:
        to_pca_subspace = all_sess_eigenvecs.xs(to_window, level='start_time').values.T

        # Compute SVD between matching subspaces
        M = from_pca_subspace.T @ to_pca_subspace
        from_s_c, S, to_s_comp_h_c = np.linalg.svd(M)
        canonc_angles = np.arccos(np.clip(S, -1, 1))
        canonc_angles = np.round(np.rad2deg(canonc_angles), 2)
        # print(canonc_angles)
        # aggr_info
        angle_distr = {
            'from_window': from_window,
            'to_window': to_window,
            'mean': np.mean(canonc_angles),
            'median': np.median(canonc_angles),
            'std': np.std(canonc_angles),
            '20_percentile': np.percentile(canonc_angles, 25),
            '80_percentile': np.percentile(canonc_angles, 75),
        }
        angle_aggr.append(pd.Series(angle_distr))
    break
        
angle_aggr_df = pd.DataFrame(angle_aggr)
print(angle_aggr_df)




In [ ]:
d = angle_aggr_df.pivot(index='from_window', columns='to_window', values='mean')
d

In [ ]:
plt.close('all')
d = angle_aggr_df.pivot(index='from_window', columns='to_window', values='mean')
plt.imshow(d, cmap='viridis', origin='lower')
plt.colorbar(label='Mean Canonical Angle (degrees)')
# angle_aggr_df.pivot(index='from_window', columns='to_window', values='mean')
# t = session_t_start.get_level_values('global_t')
# draw seperater lines
for i,st in enumerate(session_seps):
    plt.axvline(x=st, color='gray', linestyle='--', alpha=.4, linewidth=1)
    plt.axhline(y=st, color='gray', linestyle='--', alpha=.4, linewidth=1)
    plt.text(st+1, 2, f"S{i:02}", color='black', fontsize=6, ha='left', va='bottom', rotation=90)

plt.tick_params(labelleft=False, labelbottom=False)
session_seps